# Train GIT LoRA Baseline

This notebook fine-tunes `microsoft/git-base-vqav2` with LoRA adapters on the processed multi-task VQA data. Use it as a second baseline after BLIP LoRA.

## 1. Colab Repository Setup

Run this cell first in Colab. It clones or updates the repository and changes the working directory to the repo root.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)

    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import torch

try:
    import torchao
    torchao_version = getattr(torchao, "__version__", "0.0.0")
    major, minor, *_ = [int(part) for part in torchao_version.split(".")[:2]]
    if (major, minor) < (0, 16):
        print(f"Removing incompatible torchao=={torchao_version} before importing PEFT.")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
        for module_name in list(sys.modules):
            if module_name == "torchao" or module_name.startswith("torchao."):
                del sys.modules[module_name]
except ModuleNotFoundError:
    pass

try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

from peft import LoraConfig, get_peft_model
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoProcessor, GitForCausalLM

torch.__version__

## 3. Config

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "microsoft/git-base-vqav2"
LOCAL_CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/git_lora_baseline_sample"
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/git_lora_baseline_sample")
USE_GOOGLE_DRIVE_CHECKPOINT = True

MAX_SAMPLES = 100
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 4
SEED = 42
TEXT_MAX_LENGTH = 96

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["query", "value"]

DATA_LIMITS = {"docvqa": 1667, "chartqa": 1667, "textvqa": 1666}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
DEVICE

## 4. Checkpoint Storage

In [ ]:
CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 5. Prepare Data If Needed

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [sys.executable, str(PROJECT_ROOT / "scripts/build_multitask_data.py"), "--split", "validation"]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)
else:
    print("Processed data already exists. Skipping data preparation.")

## 6. Load Records

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

random.shuffle(records)
records = records[:MAX_SAMPLES]

len(records), records[0]

## 7. Dataset and Collate Function

In [ ]:
class GitVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        answer = example["answers"][0] if example["answers"] else ""
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answer": answer,
        }


processor = AutoProcessor.from_pretrained(MODEL_NAME)


def build_prompt(question: str) -> str:
    return f"question: {question} answer:"


def collate_fn(batch):
    images = [Image.open(PROJECT_ROOT / item["image_path"]).convert("RGB") for item in batch]
    prompts = [build_prompt(item["question"]) for item in batch]
    answers = [item["answer"] for item in batch]
    texts = [f"{prompt} {answer}" for prompt, answer in zip(prompts, answers)]

    inputs = processor(
        images=images,
        text=texts,
        padding=True,
        truncation=True,
        max_length=TEXT_MAX_LENGTH,
        return_tensors="pt",
    )

    labels = inputs["input_ids"].clone()
    for row_index, prompt in enumerate(prompts):
        prompt_ids = processor.tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=TEXT_MAX_LENGTH,
        )["input_ids"]
        labels[row_index, : len(prompt_ids)] = -100

    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100
    inputs["labels"] = labels
    return inputs


train_dataset = GitVQAFinetuneDataset(records)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
len(train_dataset)

## 8. Inspect One Batch

In [ ]:
debug_batch = next(iter(train_loader))
for key, value in debug_batch.items():
    print(key, value.shape, value.dtype)

valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
print("label min/max:", int(valid_labels.min()), int(valid_labels.max()))
print("tokenizer vocab size:", processor.tokenizer.vocab_size)
assert int(valid_labels.min()) >= 0
assert int(valid_labels.max()) < processor.tokenizer.vocab_size
print("Batch check passed.")

## 9. Free GPU Memory

In [ ]:
for name in ["model", "optimizer", "scaler"]:
    if name in globals():
        del globals()[name]

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(torch.cuda.memory_summary(device=None, abbreviated=True))

## 10. Load GIT with LoRA

In [ ]:
model = GitForCausalLM.from_pretrained(MODEL_NAME)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.to(DEVICE)
model.train()

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE)

model.print_trainable_parameters()
sum(parameter.numel() for parameter in trainable_parameters)

## 11. Train

In [ ]:
if DEVICE == "cuda":
    torch.cuda.empty_cache()

autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
loss_history = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}

        with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss / GRADIENT_ACCUMULATION_STEPS).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)

        if step % 10 == 0 or step == 1:
            print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")

len(loss_history), loss_history[:5], loss_history[-5:]

## 12. Save Adapter Checkpoint

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)
print(f"Saved GIT LoRA checkpoint to {CHECKPOINT_DIR}")

## 13. Quick Inference Check

In [ ]:
model.eval()
example = records[0]
image = Image.open(PROJECT_ROOT / example["image_path"]).convert("RGB")
prompt = build_prompt(example["question"])
inputs = processor(images=image, text=prompt, return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=20)

prediction = processor.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
print("question:", example["question"])
print("prediction:", prediction)
print("answers:", example["answers"])